# DMET-SSTKG — Source-Validation Motif-Loss Sensitivity Audit

This notebook reproduces the source-validation one-factor-at-a-time (OFAT)
sensitivity audit used to justify the final motif-extractor coefficients:

```text
lambda_sig = 0.02
lambda_seq = 0.20
beta_push  = 0.50
margin     = 0.70
```

Every candidate starts from the same source phase-classifier checkpoint. One
coefficient is varied while all others remain at the retained reference values.
Candidates are ranked by validation Macro-F1 error, defined as `1 - Macro F1`.
Weighted total loss is not used for ranking because its numerical scale changes
when the loss coefficients change.

## 1. Required Environment

```bash
pip install torch==2.4.1 torch-geometric pandas numpy matplotlib scikit-learn tqdm
```

Select the PyTorch build appropriate for the local CPU or CUDA environment.

In [ ]:
import copy
import gc
import json
import os
import random
import shutil
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GATv2Conv
from tqdm import tqdm

## 2. Repository Paths

The paths are repository-relative examples and contain no user-specific
directory, video name, or case name.

- `BASE_PATH`: source S-STKG case directories.
- `BIN_PATH`: binary cache shared with the phase-classifier and motif stages.
- `PHASE_RESULTS_DIR`: selected source phase-classifier run.
- `PHASE_CKPT`: source phase-classifier initialization checkpoint.
- `SPLIT_NPZ`: exact saved case-disjoint split.
- `OUTPUT_ROOT`: incremental sensitivity-audit outputs.

Replace only `selected_phase_run` with the phase-classifier run used by the
final motif extractor.

In [ ]:
BASE_PATH = Path("data/source_sstkg")
BIN_PATH = Path("outputs/phase_classifier_cache")

PHASE_RESULTS_DIR = Path(
    "outputs/phase_classifier_runs/selected_phase_run"
)
PHASE_CKPT = PHASE_RESULTS_DIR / "best_sstkg_phase_classifier.pt"
SPLIT_NPZ = PHASE_RESULTS_DIR / "data_split.npz"

FORCE_REPROCESS_CACHE = False
CHUNK_SIZE = 100
NUM_PHASES = 8
HIDDEN_DIM = 128
MOTIF_DIM = 128

LABEL_SMOOTHING = 0.05
MOTIF_LR = 5e-4
BACKBONE_LR = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

REMOVE_CASE_ID_FEATURE = True
REMOVE_G_RATIO_FEATURE = False
REMOVE_P_RATIO_FEATURE = False
CASE_ID_COL = 7
G_RATIO_COL = 8
P_RATIO_COL = 9

PHASE_NAMES = [
    "Trocar Placement",
    "Preparation",
    "Calot Triangle Dissection",
    "Clipping and Cutting",
    "Gallbladder Dissection",
    "Gallbladder Packaging",
    "Cleaning and Coagulation",
    "Gallbladder Retraction",
]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

## 3. Sensitivity Design

Stage 1 evaluates 16 unique reference-centered configurations: the reference,
four additional values for `lambda_sig`, four for `lambda_seq`, three for
`beta_push`, and four for `margin`. The candidate ranges were selected to keep
the signature-space regularization and auxiliary sequence objective subordinate
to the primary motif-classification loss while examining meaningful variation
in temporal regularization and signature-space collapse control. Small
`lambda_sig` values assess signature regularization without allowing it to
dominate motif discrimination; moderate `lambda_seq` values examine how
strongly the training-only sequence branch should preserve phase alignment;
the `beta_push` range tests repulsion strengths that can limit signature
collapse without excessive within-fragment separation; and margin values from
0.60 to 0.80 examine moderate cosine-similarity tolerances for the push term.

The optional confirmation stage is disabled by default and can evaluate the
strongest Stage-1 alternatives over seeds 42, 43, and 44.

In [ ]:
REFERENCE = {
    "lambda_sig": 0.02,
    "lambda_seq": 0.20,
    "beta_push": 0.50,
    "margin": 0.70,
}

SEARCH_SPACE = {
    "lambda_sig": [0.01, 0.02, 0.03, 0.04, 0.05],
    "lambda_seq": [0.10, 0.20, 0.30, 0.40, 0.50],
    "beta_push": [0.20, 0.40, 0.50, 0.60],
    "margin": [0.60, 0.65, 0.70, 0.75, 0.80],
}

# Stage 1 reproduces the source-validation OFAT screen reported in the paper.
SCREEN_SEEDS = [42]
SCREEN_EPOCHS = 3
SCREEN_TRAIN_FRACTION = 0.35
SCREEN_PATIENCE = 2

# Optional lightweight three-seed confirmation of the reference and nearest
# Stage-1 alternatives. Disabled by default.
RUN_CONFIRMATION = False
CONFIRM_TOP_VALUES_PER_PARAMETER = 2
CONFIRM_SEEDS = [42, 43, 44]
CONFIRM_EPOCHS = 3
CONFIRM_TRAIN_FRACTION = 0.35
CONFIRM_PATIENCE = 2

RESUME = True
SAVE_CONFIRM_CHECKPOINTS = False
OUTPUT_ROOT = Path("outputs/motif_loss_sensitivity_audit")

## 4. Reproducibility Helpers

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def safe_label_to_zero_based(raw_label: int, num_classes: int = 8) -> int:
    raw_label = int(raw_label)
    if 1 <= raw_label <= num_classes:
        return raw_label - 1
    if 0 <= raw_label < num_classes:
        return raw_label
    raise ValueError(f"Unexpected phase label value: {raw_label}")


def config_id(config: dict) -> str:
    return (
        f"lsig_{config['lambda_sig']:.2f}"
        f"__lseq_{config['lambda_seq']:.2f}"
        f"__bpush_{config['beta_push']:.2f}"
        f"__m_{config['margin']:.2f}"
    )


def same_float(a: float, b: float, atol: float = 1e-12) -> bool:
    return abs(float(a) - float(b)) <= atol


# =============================================================================

## 5. Reusing the Phase-Classifier S-STKG Cache

The phase-classifier stage converts the text-based S-STKG files into a binary
cache of temporally ordered graph sequences. This sensitivity audit reuses that
same cache because the saved `data_split.npz` file refers to fragments by their
cache indices. Reusing the cache therefore preserves the exact train,
validation, and source-test assignments established during phase-classifier
training.

The cache should be rebuilt only when the source S-STKG files or the input
feature configuration have changed. Physical node classes are read unchanged
from the constructed S-STKG files. The local `case_id` field is retained only
as fragment metadata and is set to zero in the node features before model
input.

In [ ]:
class SurgicalTurboDataset(Dataset):
    def __init__(self, root_dir, binary_dir, chunk_size=100, force_reprocess=False):
        super().__init__(root_dir)
        self.root_dir = root_dir
        self.binary_dir = binary_dir
        self.chunk_size = chunk_size

        if force_reprocess:
            print(f"[INFO] Removing cache: {self.binary_dir}")
            if os.path.exists(self.binary_dir):
                shutil.rmtree(self.binary_dir)
            os.makedirs(self.binary_dir, exist_ok=True)
            self._convert_and_fragment(root_dir)
        elif not os.path.exists(self.binary_dir) or len(os.listdir(self.binary_dir)) == 0:
            print(f"[INFO] Cache missing. Building: {self.binary_dir}")
            os.makedirs(self.binary_dir, exist_ok=True)
            self._convert_and_fragment(root_dir)
        else:
            print(f"[OK] Existing cache is used: {self.binary_dir}")

        self.files = sorted(
            f for f in os.listdir(self.binary_dir) if f.endswith(".pt")
        )
        if not self.files:
            raise RuntimeError(f"Binary cache is empty: {self.binary_dir}")

    def _clean_node_features(self, f_n):
        f_n = np.asarray(f_n).copy()
        if f_n.ndim != 2 or f_n.shape[1] < 11:
            raise ValueError(
                f"Unexpected node feature shape: {f_n.shape}; expected >=11 columns."
            )
        if REMOVE_CASE_ID_FEATURE:
            f_n[:, CASE_ID_COL] = 0.0
        if REMOVE_G_RATIO_FEATURE:
            f_n[:, G_RATIO_COL] = 0.0
        if REMOVE_P_RATIO_FEATURE:
            f_n[:, P_RATIO_COL] = 0.0
        return f_n

    def _convert_and_fragment(self, root_dir):
        print("[INFO] Converting source S-STKGs into binary fragments...")
        if not os.path.exists(root_dir):
            raise FileNotFoundError(f"BASE_PATH not found: {root_dir}")

        cases = sorted(
            d for d in os.listdir(root_dir) if d.startswith("Case_")
        )
        if not cases:
            raise RuntimeError(f"No Case_ folders found under {root_dir}")

        global_idx = 0
        for case in tqdm(cases, desc="Building source motif cache"):
            case_path = os.path.join(root_dir, case)
            prefixes = {
                f.replace("_nodes.txt", "")
                for f in os.listdir(case_path)
                if f.endswith("_nodes.txt")
            }

            for pref in sorted(prefixes):
                nodes_path = os.path.join(case_path, f"{pref}_nodes.txt")
                edges_path = os.path.join(case_path, f"{pref}_edges.txt")
                edge_feat_path = os.path.join(case_path, f"{pref}_edge_features.txt")
                label_path = os.path.join(case_path, f"{pref}_labels.txt")

                required = [nodes_path, edges_path, edge_feat_path, label_path]
                if not all(os.path.exists(p) for p in required):
                    print(f"[WARN] Missing files for prefix {pref}; skipping.")
                    continue

                n_df = pd.read_csv(nodes_path)
                e_df = pd.read_csv(edges_path)
                ef_df = pd.read_csv(edge_feat_path)
                raw_label = int(pd.read_csv(label_path).iloc[0, 1])
                label = safe_label_to_zero_based(raw_label, NUM_PHASES)
                unique_grs = sorted(n_df.iloc[:, G_RATIO_COL].unique())

                for i in range(0, len(unique_grs), self.chunk_size):
                    chunk_grs = unique_grs[i : i + self.chunk_size]
                    if len(chunk_grs) < 10:
                        continue

                    graph_seq = []
                    for gr in chunk_grs:
                        f_n_raw = n_df[n_df.iloc[:, G_RATIO_COL] == gr].values
                        f_n = self._clean_node_features(f_n_raw)
                        f_e = e_df[e_df.iloc[:, 0] == gr].values
                        f_ef = ef_df[ef_df.iloc[:, 0] == gr].values

                        edge_index = (
                            torch.tensor(
                                f_e[:, 2:4].astype(np.int64), dtype=torch.long
                            )
                            .t()
                            .contiguous()
                            if len(f_e) > 0
                            else torch.zeros((2, 0), dtype=torch.long)
                        )
                        edge_attr = (
                            torch.tensor(f_ef[:, 4:7], dtype=torch.float)
                            if len(f_ef) > 0
                            else torch.zeros((0, 3), dtype=torch.float)
                        )

                        data = Data(
                            x=torch.tensor(f_n, dtype=torch.float),
                            edge_index=edge_index,
                            edge_attr=edge_attr,
                            y=torch.tensor([label], dtype=torch.long),
                        )
                        data.is_v = data.x[:, 10] == 1
                        graph_seq.append(data)

                    torch.save(
                        {
                            "graph_seq": graph_seq,
                            "metadata": {
                                "case_id": str(case),
                                "prefix": str(pref),
                                "chunk_start_gr": float(chunk_grs[0]),
                                "chunk_end_gr": float(chunk_grs[-1]),
                                "phase_label_id": int(label),
                                "raw_phase_label": int(raw_label),
                            },
                        },
                        os.path.join(
                            self.binary_dir, f"frag_{global_idx:06d}.pt"
                        ),
                    )
                    global_idx += 1

        print(f"[OK] Binary fragments created: {global_idx}")

    def len(self):
        return len(self.files)

    def _load_raw(self, idx):
        return torch.load(
            os.path.join(self.binary_dir, self.files[int(idx)]),
            weights_only=False,
        )

    def get_with_meta(self, idx):
        obj = self._load_raw(idx)
        if isinstance(obj, dict) and "graph_seq" in obj:
            meta = dict(obj.get("metadata", {}) or {})
            meta["fragment_file"] = self.files[int(idx)]
            return obj["graph_seq"], meta
        return obj, {
            "fragment_file": self.files[int(idx)],
            "case_id": "UNKNOWN",
        }

    def get(self, idx):
        seq, _ = self.get_with_meta(idx)
        return seq


# =============================================================================

## 6. Saved Split and Phase-Stratified Screen Subset

The complete source validation split is retained. Stage 1 uses a
phase-stratified fraction of the source training split to reduce cost. Source-test
indices are not evaluated.

In [ ]:
def load_saved_split(split_npz, dataset_len):
    if not os.path.exists(split_npz):
        raise FileNotFoundError(f"SPLIT_NPZ not found: {split_npz}")

    s = np.load(split_npz)
    train_idx = s["train_idx"].astype(int)
    val_idx = s["val_idx"].astype(int)
    test_idx = s["test_idx"].astype(int)

    all_idx = np.concatenate([train_idx, val_idx, test_idx])
    if all_idx.min() < 0 or all_idx.max() >= dataset_len:
        raise ValueError(
            f"Split incompatible with dataset length={dataset_len}; "
            f"min={all_idx.min()}, max={all_idx.max()}."
        )
    if (
        set(train_idx) & set(val_idx)
        or set(train_idx) & set(test_idx)
        or set(val_idx) & set(test_idx)
    ):
        raise ValueError("Split overlap detected.")

    return train_idx, val_idx, test_idx


def fragment_label(dataset, idx: int) -> int:
    seq = dataset.get(int(idx))
    if not seq:
        raise RuntimeError(f"Empty graph sequence at index {idx}")
    return int(seq[0].y.item())


def stratified_fraction_indices(
    dataset,
    indices,
    fraction: float,
    seed: int,
):
    indices = np.asarray(indices, dtype=int)
    if not 0.0 < fraction <= 1.0:
        raise ValueError("fraction must be in (0, 1].")
    if same_float(fraction, 1.0):
        return indices.copy()

    labels = np.array([fragment_label(dataset, idx) for idx in indices])
    rng = np.random.default_rng(seed)
    selected = []

    for phase in sorted(np.unique(labels)):
        phase_idx = indices[labels == phase].copy()
        rng.shuffle(phase_idx)
        n_take = max(1, int(round(len(phase_idx) * fraction)))
        selected.extend(phase_idx[:n_take].tolist())

    selected = np.asarray(selected, dtype=int)
    rng.shuffle(selected)
    return selected


def compute_motif_class_weights(
    dataset,
    indices,
    motif_window=16,
    motif_stride=8,
    num_classes=8,
):
    counts = np.zeros(num_classes, dtype=np.float64)
    for idx in indices:
        seq = dataset.get(int(idx))
        label = int(seq[0].y.item())
        t_len = len(seq)
        n_motifs = (
            1
            if t_len < motif_window
            else len(range(0, t_len - motif_window + 1, motif_stride))
        )
        counts[label] += n_motifs

    counts = np.clip(counts, a_min=1, a_max=None)
    weights = 1.0 / counts
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float)


# =============================================================================

## 7. Phase-Initialized Motif Model

The model matches the final motif-extractor architecture and partial-freezing
policy. The checkpoint loader accepts both the development module names and the
GitHub phase-classifier module names, while allowing missing keys only for the
new motif-specific layers.

In [ ]:
class RobustMosaicGNN(nn.Module):
    def __init__(
        self,
        in_channels=11,
        edge_dim=3,
        hidden_dim=128,
        num_phases=8,
        motif_dim=128,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.conv1 = GATv2Conv(in_channels, hidden_dim, edge_dim=edge_dim)
        self.conv2 = GATv2Conv(hidden_dim, hidden_dim, edge_dim=edge_dim)
        self.context_attn = nn.Linear(hidden_dim * 2, 1)
        self.rnn = nn.GRU(
            hidden_dim, hidden_dim, batch_first=True, bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 3, num_phases)
        self.motif_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, motif_dim),
        )
        self.motif_cls = nn.Linear(motif_dim, num_phases)
        self.window_fuse = nn.Linear(hidden_dim * 2, hidden_dim)
        self.motif_window = 16
        self.motif_stride = 8

    def encode_frames(self, graph_seq):
        frame_embs = []
        for data in graph_seq:
            x = self.conv1(
                data.x, data.edge_index, data.edge_attr
            ).relu()
            x = self.conv2(
                x, data.edge_index, data.edge_attr
            ).relu()

            mean_emb = x.mean(dim=0, keepdim=True)
            v_node_emb = (
                x[data.is_v].mean(dim=0, keepdim=True)
                if data.is_v.any()
                else mean_emb
            )
            attn_weight = torch.sigmoid(
                self.context_attn(
                    torch.cat([mean_emb, v_node_emb], dim=-1)
                )
            )
            frame_embs.append(
                attn_weight * v_node_emb
                + (1.0 - attn_weight) * mean_emb
            )
        return torch.stack(frame_embs).squeeze(1)

    def apply_temporal_modeling(self, frame_tensor):
        rnn_out, h_n = self.rnn(frame_tensor.unsqueeze(0))
        rnn_out = rnn_out.squeeze(0)
        temporal_out = 0.5 * (
            rnn_out[:, : self.hidden_dim]
            + rnn_out[:, self.hidden_dim :]
        )
        temporal_frames = 0.5 * frame_tensor + 0.5 * temporal_out
        rnn_final = torch.cat([h_n[0], h_n[1]], dim=-1)
        global_skip = frame_tensor.mean(dim=0, keepdim=True)
        seq_feature = torch.cat([rnn_final, global_skip], dim=-1)
        return temporal_frames, seq_feature

    def pool_window(self, window_tensor):
        pooled = torch.cat(
            [
                window_tensor.mean(dim=0, keepdim=True),
                window_tensor[-1].unsqueeze(0),
            ],
            dim=-1,
        )
        return self.window_fuse(pooled)

    def extract_motifs(self, frame_tensor):
        t_len = frame_tensor.shape[0]
        if t_len < self.motif_window:
            return F.normalize(
                self.motif_proj(self.pool_window(frame_tensor)), dim=-1
            )

        windows = [
            self.pool_window(
                frame_tensor[start : start + self.motif_window]
            )
            for start in range(
                0,
                t_len - self.motif_window + 1,
                self.motif_stride,
            )
        ]
        return F.normalize(
            self.motif_proj(torch.cat(windows, dim=0)), dim=-1
        )

    def forward(self, graph_seq):
        frames = self.encode_frames(graph_seq)
        temporal_frames, seq_feature = self.apply_temporal_modeling(frames)
        sigs = self.extract_motifs(temporal_frames)
        return {
            "motif_logits": self.motif_cls(sigs),
            "motif_sigs": sigs,
            "seq_logits": self.fc(seq_feature),
        }


def apply_partial_unfreeze(model):
    for parameter in model.parameters():
        parameter.requires_grad = False

    trainable_tokens = (
        "motif_",
        "conv2",
        "context_attn",
        "rnn",
        "window_fuse",
        "fc",
    )
    for name, parameter in model.named_parameters():
        if any(token in name for token in trainable_tokens):
            parameter.requires_grad = True


def build_optimizer(model):
    motif_params = []
    backbone_params = []

    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if (
            "motif_" in name
            or "window_fuse" in name
            or "fc" in name
        ):
            motif_params.append(parameter)
        else:
            backbone_params.append(parameter)

    return torch.optim.AdamW(
        [
            {"params": motif_params, "lr": MOTIF_LR},
            {"params": backbone_params, "lr": BACKBONE_LR},
        ],
        weight_decay=WEIGHT_DECAY,
    )


def _phase_state_dict_for_audit_model(state_dict):
    """Support both development and GitHub phase-checkpoint module names."""
    prefix_map = {
        "graph_conv_1.": "conv1.",
        "graph_conv_2.": "conv2.",
        "virtual_context_gate.": "context_attn.",
        "temporal_encoder.": "rnn.",
        "phase_head.": "fc.",
    }

    converted = {}
    for key, value in state_dict.items():
        converted_key = key
        for source_prefix, target_prefix in prefix_map.items():
            if key.startswith(source_prefix):
                converted_key = target_prefix + key[len(source_prefix):]
                break
        converted[converted_key] = value
    return converted


def initialize_from_phase_checkpoint(seed: int):
    # The seed is reset before construction so all coefficient candidates use
    # identical motif-layer initialization within the same seed.
    set_seed(seed)
    model = RobustMosaicGNN(
        hidden_dim=HIDDEN_DIM,
        num_phases=NUM_PHASES,
        motif_dim=MOTIF_DIM,
    ).to(DEVICE)

    checkpoint = torch.load(
        PHASE_CKPT,
        map_location=DEVICE,
        weights_only=False,
    )
    raw_state_dict = checkpoint.get("model_state_dict", checkpoint)
    state_dict = _phase_state_dict_for_audit_model(raw_state_dict)
    message = model.load_state_dict(state_dict, strict=False)

    allowed_missing_prefixes = (
        "motif_proj.",
        "motif_cls.",
        "window_fuse.",
    )
    unexpected_missing = [
        key for key in message.missing_keys
        if not key.startswith(allowed_missing_prefixes)
    ]
    if unexpected_missing:
        raise RuntimeError(
            "Shared phase-model parameters are missing: "
            f"{unexpected_missing}"
        )
    if message.unexpected_keys:
        raise RuntimeError(
            "Unexpected phase-checkpoint parameters: "
            f"{message.unexpected_keys}"
        )

    apply_partial_unfreeze(model)
    return model


# =============================================================================

## 8. Parameterized Motif Objective

The varied objective is:

```text
motif classification
+ lambda_sig × signature regularization
+ lambda_seq × auxiliary sequence classification
```

The signature regularizer combines adjacent-window consistency with the
`beta_push`-weighted repulsion term above the selected margin.

In [ ]:
def motif_signature_loss(
    sigs,
    beta_push: float,
    margin: float,
):
    if sigs.size(0) <= 1:
        zero = torch.tensor(0.0, device=sigs.device)
        return zero, zero, zero

    pull = (
        1.0
        - F.cosine_similarity(
            sigs[:-1],
            sigs[1:],
            dim=-1,
        )
    ).mean()

    sim_mat = sigs @ sigs.t()
    mask = ~torch.eye(
        sim_mat.size(0),
        device=sigs.device,
        dtype=torch.bool,
    )
    off_diag = sim_mat[mask]
    push = (
        torch.relu(off_diag - float(margin)).mean()
        if off_diag.numel() > 0
        else torch.tensor(0.0, device=sigs.device)
    )

    total = pull + float(beta_push) * push
    return total, pull, push


def compute_total_motif_loss(
    out,
    motif_target,
    seq_target,
    criterion,
    config,
):
    motif_cls = criterion(out["motif_logits"], motif_target)
    sig_total, sig_pull, sig_push = motif_signature_loss(
        out["motif_sigs"],
        beta_push=config["beta_push"],
        margin=config["margin"],
    )
    seq_cls = criterion(out["seq_logits"], seq_target)

    total = (
        motif_cls
        + float(config["lambda_sig"]) * sig_total
        + float(config["lambda_seq"]) * seq_cls
    )

    return {
        "total": total,
        "motif_cls": motif_cls,
        "sig_total": sig_total,
        "sig_pull": sig_pull,
        "sig_push": sig_push,
        "seq_cls": seq_cls,
    }


# =============================================================================

## 9. Validation Metrics

The primary ranking measure is validation Macro-F1 error. Validation motif
classification cross-entropy is used only to break exact Macro-F1 ties.

In [ ]:
def compute_basic_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    _, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )
    return {
        "accuracy": float(accuracy),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "macro_f1_error": float(1.0 - macro_f1),
        "weighted_f1": float(weighted_f1),
    }


def evaluate_validation(
    model,
    dataset,
    val_idx,
    criterion,
    config,
):
    model.eval()
    component_names = [
        "total",
        "motif_cls",
        "sig_total",
        "sig_pull",
        "sig_push",
        "seq_cls",
    ]
    sums = {name: [] for name in component_names}
    y_true = []
    y_pred = []

    with torch.no_grad():
        for idx in val_idx:
            seq = [data.to(DEVICE) for data in dataset.get(int(idx))]
            out = model(seq)
            motif_target = seq[0].y.to(DEVICE).repeat(
                out["motif_logits"].size(0)
            )
            seq_target = seq[0].y.to(DEVICE)

            losses = compute_total_motif_loss(
                out,
                motif_target,
                seq_target,
                criterion,
                config,
            )
            for name in component_names:
                sums[name].append(float(losses[name].item()))

            predictions = (
                out["motif_logits"]
                .argmax(dim=1)
                .detach()
                .cpu()
                .numpy()
            )
            y_pred.extend(predictions.tolist())
            y_true.extend(
                [int(seq[0].y.item())] * len(predictions)
            )

    metrics = compute_basic_metrics(
        np.asarray(y_true),
        np.asarray(y_pred),
    )
    for name in component_names:
        metrics[f"val_{name}_loss"] = (
            float(np.mean(sums[name])) if sums[name] else np.nan
        )
    return metrics


# =============================================================================

## 10. OFAT Candidate Construction

In [ ]:
def build_ofat_configs():
    """
    Returns 16 unique configurations:
        1 reference
        4 additional lambda_sig values
        4 additional lambda_seq values
        3 additional beta_push values
        4 additional margin values
    """
    configs = [dict(REFERENCE)]
    for parameter, values in SEARCH_SPACE.items():
        for value in values:
            if same_float(value, REFERENCE[parameter]):
                continue
            candidate = dict(REFERENCE)
            candidate[parameter] = float(value)
            configs.append(candidate)

    unique = {}
    for config in configs:
        unique[config_id(config)] = config
    return list(unique.values())


def identify_varied_parameter(config):
    differences = [
        parameter
        for parameter in REFERENCE
        if not same_float(config[parameter], REFERENCE[parameter])
    ]
    if not differences:
        return "reference"
    if len(differences) == 1:
        return differences[0]
    return "joint"


def trial_key(stage, config, seed):
    return f"{stage}__{config_id(config)}__seed_{seed}"


# =============================================================================

## 11. Single-Trial Training

Within a seed, every coefficient configuration uses the same phase checkpoint,
new-layer initialization, and epoch-specific training order.

In [ ]:
def run_trial(
    stage,
    config,
    seed,
    dataset,
    train_idx,
    val_idx,
    class_weights,
    epochs,
    patience,
    save_checkpoint_path=None,
):
    model = initialize_from_phase_checkpoint(seed)
    optimizer = build_optimizer(model)
    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE),
        label_smoothing=LABEL_SMOOTHING,
    )

    best_metrics = None
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0
    start_time = time.time()

    # Identical epoch-specific order is used for all configurations with a
    # given seed, preserving a fair coefficient comparison.
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_rng = np.random.default_rng(seed * 1000 + epoch)
        epoch_indices = np.asarray(train_idx, dtype=int).copy()
        epoch_rng.shuffle(epoch_indices)

        for idx in tqdm(
            epoch_indices,
            desc=(
                f"{stage} | {config_id(config)} | "
                f"seed={seed} | epoch={epoch}/{epochs}"
            ),
            leave=False,
        ):
            seq = [data.to(DEVICE) for data in dataset.get(int(idx))]
            optimizer.zero_grad(set_to_none=True)
            out = model(seq)

            motif_target = seq[0].y.to(DEVICE).repeat(
                out["motif_logits"].size(0)
            )
            seq_target = seq[0].y.to(DEVICE)

            losses = compute_total_motif_loss(
                out,
                motif_target,
                seq_target,
                criterion,
                config,
            )
            losses["total"].backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=GRAD_CLIP_NORM,
            )
            optimizer.step()

        val_metrics = evaluate_validation(
            model,
            dataset,
            val_idx,
            criterion,
            config,
        )

        # Primary: higher validation Macro F1.
        # Tie-breaker: lower coefficient-independent motif classification loss.
        improved = False
        if best_metrics is None:
            improved = True
        elif val_metrics["macro_f1"] > best_metrics["macro_f1"] + 1e-12:
            improved = True
        elif (
            abs(
                val_metrics["macro_f1"]
                - best_metrics["macro_f1"]
            )
            <= 1e-12
            and val_metrics["val_motif_cls_loss"]
            < best_metrics["val_motif_cls_loss"] - 1e-12
        ):
            improved = True

        if improved:
            best_metrics = dict(val_metrics)
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"[{stage}] {config_id(config)} seed={seed} epoch={epoch} | "
            f"Val MacroF1={val_metrics['macro_f1']:.4f} | "
            f"Error={val_metrics['macro_f1_error']:.4f} | "
            f"MotifCE={val_metrics['val_motif_cls_loss']:.4f}"
        )

        if (
            patience is not None
            and patience > 0
            and epochs_without_improvement >= patience
        ):
            print(
                f"[EARLY STOP] {config_id(config)} seed={seed} "
                f"after epoch {epoch}."
            )
            break

    if best_state is None or best_metrics is None:
        raise RuntimeError("Trial completed without validation metrics.")

    if save_checkpoint_path is not None:
        torch.save(
            {
                "stage": stage,
                "seed": int(seed),
                "config": dict(config),
                "best_epoch": int(best_epoch),
                "model_state_dict": best_state,
                "validation_metrics": best_metrics,
                "phase_checkpoint": PHASE_CKPT,
                "split_npz": SPLIT_NPZ,
            },
            save_checkpoint_path,
        )

    duration_minutes = (time.time() - start_time) / 60.0

    result = {
        "trial_key": trial_key(stage, config, seed),
        "stage": stage,
        "config_id": config_id(config),
        "varied_parameter": identify_varied_parameter(config),
        "seed": int(seed),
        "lambda_sig": float(config["lambda_sig"]),
        "lambda_seq": float(config["lambda_seq"]),
        "beta_push": float(config["beta_push"]),
        "margin": float(config["margin"]),
        "best_epoch": int(best_epoch),
        "epochs_budget": int(epochs),
        "train_fragments": int(len(train_idx)),
        "val_fragments": int(len(val_idx)),
        "duration_minutes": float(duration_minutes),
        "val_macro_f1": float(best_metrics["macro_f1"]),
        "val_macro_f1_error": float(best_metrics["macro_f1_error"]),
        "val_accuracy": float(best_metrics["accuracy"]),
        "val_macro_recall": float(best_metrics["macro_recall"]),
        "val_weighted_f1": float(best_metrics["weighted_f1"]),
        "val_total_loss": float(best_metrics["val_total_loss"]),
        "val_motif_cls_loss": float(
            best_metrics["val_motif_cls_loss"]
        ),
        "val_sig_total_loss": float(
            best_metrics["val_sig_total_loss"]
        ),
        "val_sig_pull_loss": float(
            best_metrics["val_sig_pull_loss"]
        ),
        "val_sig_push_loss": float(
            best_metrics["val_sig_push_loss"]
        ),
        "val_seq_cls_loss": float(
            best_metrics["val_seq_cls_loss"]
        ),
    }

    del model
    del optimizer
    del best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# =============================================================================

## 12. Incremental Results and Parameter Curves

Every completed trial is appended immediately to `all_trials.csv`, allowing a
long audit to resume after interruption. `screen_ofat_winners.csv` is the
primary decision file.

In [ ]:
RESULT_COLUMNS = [
    "trial_key",
    "stage",
    "config_id",
    "varied_parameter",
    "seed",
    "lambda_sig",
    "lambda_seq",
    "beta_push",
    "margin",
    "best_epoch",
    "epochs_budget",
    "train_fragments",
    "val_fragments",
    "duration_minutes",
    "val_macro_f1",
    "val_macro_f1_error",
    "val_accuracy",
    "val_macro_recall",
    "val_weighted_f1",
    "val_total_loss",
    "val_motif_cls_loss",
    "val_sig_total_loss",
    "val_sig_pull_loss",
    "val_sig_push_loss",
    "val_seq_cls_loss",
]


def append_and_save_result(result, csv_path):
    csv_path = Path(csv_path)
    if csv_path.exists():
        current = pd.read_csv(csv_path)
    else:
        current = pd.DataFrame(columns=RESULT_COLUMNS)

    current = pd.concat(
        [current, pd.DataFrame([result])],
        ignore_index=True,
    )
    current = (
        current.drop_duplicates(
            subset=["trial_key"],
            keep="last",
        )
        .sort_values(
            ["stage", "config_id", "seed"],
            kind="stable",
        )
        .reset_index(drop=True)
    )
    current.to_csv(csv_path, index=False)
    return current


def summarize_stage(results, stage):
    stage_df = results[results["stage"] == stage].copy()
    if stage_df.empty:
        return pd.DataFrame()

    group_cols = [
        "config_id",
        "varied_parameter",
        "lambda_sig",
        "lambda_seq",
        "beta_push",
        "margin",
    ]
    summary = (
        stage_df.groupby(group_cols, as_index=False)
        .agg(
            n_seeds=("seed", "nunique"),
            mean_val_macro_f1=("val_macro_f1", "mean"),
            sd_val_macro_f1=("val_macro_f1", "std"),
            mean_val_macro_f1_error=("val_macro_f1_error", "mean"),
            sd_val_macro_f1_error=("val_macro_f1_error", "std"),
            mean_val_accuracy=("val_accuracy", "mean"),
            mean_val_motif_cls_loss=("val_motif_cls_loss", "mean"),
            mean_duration_minutes=("duration_minutes", "mean"),
        )
    )
    summary["sd_val_macro_f1"] = summary[
        "sd_val_macro_f1"
    ].fillna(0.0)
    summary["sd_val_macro_f1_error"] = summary[
        "sd_val_macro_f1_error"
    ].fillna(0.0)
    return summary.sort_values(
        [
            "mean_val_macro_f1_error",
            "mean_val_motif_cls_loss",
        ],
        ascending=[True, True],
        kind="stable",
    ).reset_index(drop=True)


def parameter_curve(summary, parameter):
    rows = []
    for value in SEARCH_SPACE[parameter]:
        config = dict(REFERENCE)
        config[parameter] = float(value)
        cid = config_id(config)
        match = summary[summary["config_id"] == cid]
        if match.empty:
            continue
        row = match.iloc[0].to_dict()
        row["parameter"] = parameter
        row["parameter_value"] = float(value)
        row["is_reference_value"] = same_float(
            value, REFERENCE[parameter]
        )
        rows.append(row)
    return pd.DataFrame(rows)


def determine_ofat_winners(summary):
    winner_rows = []
    curve_rows = []

    for parameter in SEARCH_SPACE:
        curve = parameter_curve(summary, parameter)
        if curve.empty:
            continue

        curve = curve.sort_values(
            [
                "mean_val_macro_f1_error",
                "mean_val_motif_cls_loss",
            ],
            ascending=[True, True],
            kind="stable",
        ).reset_index(drop=True)
        winner = curve.iloc[0]

        winner_rows.append(
            {
                "parameter": parameter,
                "reference_value": float(REFERENCE[parameter]),
                "winner_value": float(winner["parameter_value"]),
                "reference_is_winner": bool(
                    same_float(
                        winner["parameter_value"],
                        REFERENCE[parameter],
                    )
                ),
                "winner_mean_val_macro_f1": float(
                    winner["mean_val_macro_f1"]
                ),
                "winner_mean_val_macro_f1_error": float(
                    winner["mean_val_macro_f1_error"]
                ),
                "winner_mean_val_motif_cls_loss": float(
                    winner["mean_val_motif_cls_loss"]
                ),
                "winner_config_id": str(winner["config_id"]),
            }
        )
        curve_rows.append(curve)

    winners = pd.DataFrame(winner_rows)
    curves = (
        pd.concat(curve_rows, ignore_index=True)
        if curve_rows
        else pd.DataFrame()
    )
    return winners, curves


def select_confirmation_configs(screen_summary):
    selected_ids = {config_id(REFERENCE)}

    for parameter in SEARCH_SPACE:
        curve = parameter_curve(screen_summary, parameter)
        if curve.empty:
            continue
        curve = curve.sort_values(
            [
                "mean_val_macro_f1_error",
                "mean_val_motif_cls_loss",
            ],
            ascending=[True, True],
            kind="stable",
        )
        top = curve.head(CONFIRM_TOP_VALUES_PER_PARAMETER)
        selected_ids.update(top["config_id"].tolist())

    all_ofat = {
        config_id(config): config
        for config in build_ofat_configs()
    }
    return [
        all_ofat[cid]
        for cid in sorted(selected_ids)
        if cid in all_ofat
    ]


def plot_parameter_curves(curves, output_dir, stage_label):
    if curves.empty:
        return

    label_map = {
        "lambda_sig": r"$\lambda_{\mathrm{sig}}$",
        "lambda_seq": r"$\lambda_{\mathrm{seq}}$",
        "beta_push": r"$\beta_{\mathrm{push}}$",
        "margin": r"$m$",
    }

    for parameter in SEARCH_SPACE:
        subset = curves[curves["parameter"] == parameter].copy()
        if subset.empty:
            continue
        subset = subset.sort_values("parameter_value")

        fig, ax = plt.subplots(figsize=(6.4, 4.6))
        ax.plot(
            subset["parameter_value"],
            subset["mean_val_macro_f1_error"],
            marker="o",
        )
        ref_row = subset[subset["is_reference_value"]]
        if not ref_row.empty:
            ax.scatter(
                ref_row["parameter_value"],
                ref_row["mean_val_macro_f1_error"],
                marker="*",
                s=160,
                label="Reference value",
                zorder=5,
            )
        ax.set_xlabel(label_map[parameter])
        ax.set_ylabel("Validation Macro-F1 error (1 - Macro F1)")
        ax.set_title(f"{stage_label}: {label_map[parameter]} sensitivity")
        ax.grid(True, alpha=0.25)
        ax.legend()
        fig.tight_layout()
        fig.savefig(
            os.path.join(
                output_dir,
                f"{stage_label}_{parameter}_sensitivity.png",
            ),
            dpi=300,
        )
        plt.close(fig)


# =============================================================================

## 13. Run the Audit

Stage 1 runs the 16-configuration OFAT screen. Set `RUN_CONFIRMATION=True` only
when the optional lightweight three-seed confirmation is required.

In [ ]:
def run_sensitivity_audit():
    if not os.path.exists(PHASE_CKPT):
        raise FileNotFoundError(f"PHASE_CKPT not found: {PHASE_CKPT}")
    if not os.path.exists(SPLIT_NPZ):
        raise FileNotFoundError(f"SPLIT_NPZ not found: {SPLIT_NPZ}")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = OUTPUT_ROOT if RESUME else Path(f"{OUTPUT_ROOT}_{timestamp}")
    os.makedirs(output_dir, exist_ok=True)

    results_csv = os.path.join(output_dir, "all_trials.csv")
    existing = (
        pd.read_csv(results_csv)
        if RESUME and os.path.exists(results_csv)
        else pd.DataFrame(columns=RESULT_COLUMNS)
    )
    completed_keys = set(
        existing["trial_key"].astype(str).tolist()
        if not existing.empty
        else []
    )

    print("=" * 88)
    print("DMET-SSTKG MOTIF-LOSS SENSITIVITY AUDIT")
    print("=" * 88)
    print("[DEVICE]", DEVICE)
    print("[PHASE CHECKPOINT]", PHASE_CKPT)
    print("[SPLIT]", SPLIT_NPZ)
    print("[OUTPUT]", output_dir)
    print("[REFERENCE]", REFERENCE)

    dataset = SurgicalTurboDataset(
        BASE_PATH,
        BIN_PATH,
        chunk_size=CHUNK_SIZE,
        force_reprocess=FORCE_REPROCESS_CACHE,
    )
    train_idx, val_idx, test_idx = load_saved_split(
        SPLIT_NPZ,
        len(dataset),
    )

    # Test indices are loaded only to validate split bounds and overlap.
    # No source-test graph or label is evaluated in this audit.
    del test_idx

    # Test indices are intentionally loaded only for overlap validation and are
    # never evaluated in this script.
    screen_train_idx = stratified_fraction_indices(
        dataset,
        train_idx,
        SCREEN_TRAIN_FRACTION,
        seed=42,
    )
    class_weights = compute_motif_class_weights(
        dataset,
        train_idx,
        motif_window=16,
        motif_stride=8,
        num_classes=NUM_PHASES,
    )

    design_manifest = {
        "reference": REFERENCE,
        "search_space": SEARCH_SPACE,
        "screen": {
            "design": "reference-centered one-factor-at-a-time",
            "n_unique_configs": len(build_ofat_configs()),
            "seeds": SCREEN_SEEDS,
            "epochs": SCREEN_EPOCHS,
            "train_fraction": SCREEN_TRAIN_FRACTION,
            "train_fragments": int(len(screen_train_idx)),
            "validation_fragments": int(len(val_idx)),
            "selection_metric": "validation Macro-F1 error",
            "tie_breaker": "validation motif classification CE",
        },
        "confirmation": {
            "enabled": RUN_CONFIRMATION,
            "top_values_per_parameter": CONFIRM_TOP_VALUES_PER_PARAMETER,
            "seeds": CONFIRM_SEEDS,
            "epochs": CONFIRM_EPOCHS,
            "train_fraction": CONFIRM_TRAIN_FRACTION,
        },
        "test_split_used_for_selection": False,
        "initialization": "same phase-classifier checkpoint for every trial",
        "phase_checkpoint": PHASE_CKPT,
        "split_npz": SPLIT_NPZ,
    }
    with open(
        os.path.join(output_dir, "audit_manifest.json"),
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(design_manifest, file, indent=2)

    # -------------------------------------------------------------------------
    # Stage 1: all 16 OFAT configurations
    # -------------------------------------------------------------------------
    print("\n[STAGE 1] Fast OFAT screen")
    for config in build_ofat_configs():
        for seed in SCREEN_SEEDS:
            key = trial_key("screen", config, seed)
            if key in completed_keys:
                print("[RESUME] Skipping completed:", key)
                continue

            result = run_trial(
                stage="screen",
                config=config,
                seed=seed,
                dataset=dataset,
                train_idx=screen_train_idx,
                val_idx=val_idx,
                class_weights=class_weights,
                epochs=SCREEN_EPOCHS,
                patience=SCREEN_PATIENCE,
            )
            existing = append_and_save_result(result, results_csv)
            completed_keys.add(key)

    all_results = pd.read_csv(results_csv)
    screen_summary = summarize_stage(all_results, "screen")
    screen_summary.to_csv(
        os.path.join(output_dir, "screen_summary.csv"),
        index=False,
    )

    screen_winners, screen_curves = determine_ofat_winners(
        screen_summary
    )
    screen_winners.to_csv(
        os.path.join(output_dir, "screen_ofat_winners.csv"),
        index=False,
    )
    screen_curves.to_csv(
        os.path.join(output_dir, "screen_ofat_curves.csv"),
        index=False,
    )
    plot_parameter_curves(
        screen_curves,
        output_dir,
        stage_label="screen",
    )

    print("\n[STAGE 1 WINNERS]")
    print(screen_winners.to_string(index=False))

    # -------------------------------------------------------------------------
    # Stage 2: higher-fidelity confirmation of the top two values per parameter
    # -------------------------------------------------------------------------
    if RUN_CONFIRMATION:
        confirm_configs = select_confirmation_configs(screen_summary)
        confirm_train_idx = stratified_fraction_indices(
            dataset,
            train_idx,
            CONFIRM_TRAIN_FRACTION,
            seed=42,
        )

        print(
            f"\n[STAGE 2] Confirmation: "
            f"{len(confirm_configs)} unique configurations"
        )

        checkpoint_dir = os.path.join(
            output_dir,
            "confirmation_checkpoints",
        )
        if SAVE_CONFIRM_CHECKPOINTS:
            os.makedirs(checkpoint_dir, exist_ok=True)

        for config in confirm_configs:
            for seed in CONFIRM_SEEDS:
                key = trial_key("confirm", config, seed)
                if key in completed_keys:
                    print("[RESUME] Skipping completed:", key)
                    continue

                save_path = None
                if SAVE_CONFIRM_CHECKPOINTS:
                    save_path = os.path.join(
                        checkpoint_dir,
                        f"{config_id(config)}__seed_{seed}.pt",
                    )

                result = run_trial(
                    stage="confirm",
                    config=config,
                    seed=seed,
                    dataset=dataset,
                    train_idx=confirm_train_idx,
                    val_idx=val_idx,
                    class_weights=class_weights,
                    epochs=CONFIRM_EPOCHS,
                    patience=CONFIRM_PATIENCE,
                    save_checkpoint_path=save_path,
                )
                existing = append_and_save_result(
                    result,
                    results_csv,
                )
                completed_keys.add(key)

        all_results = pd.read_csv(results_csv)
        confirm_summary = summarize_stage(
            all_results,
            "confirm",
        )
        confirm_summary.to_csv(
            os.path.join(output_dir, "confirm_summary.csv"),
            index=False,
        )

        confirm_winners, confirm_curves = determine_ofat_winners(
            confirm_summary
        )
        # A curve may be incomplete because only the top Stage-1 values were
        # promoted. The file therefore reports winners only among promoted values.
        confirm_winners.to_csv(
            os.path.join(
                output_dir,
                "confirm_promoted_value_winners.csv",
            ),
            index=False,
        )
        confirm_curves.to_csv(
            os.path.join(
                output_dir,
                "confirm_promoted_value_curves.csv",
            ),
            index=False,
        )
        plot_parameter_curves(
            confirm_curves,
            output_dir,
            stage_label="confirm",
        )

        print("\n[STAGE 2 PROMOTED-VALUE WINNERS]")
        print(confirm_winners.to_string(index=False))

    # -------------------------------------------------------------------------
    # Final decision note
    # -------------------------------------------------------------------------
    final_note_path = os.path.join(
        output_dir,
        "HOW_TO_INTERPRET_RESULTS.txt",
    )
    with open(final_note_path, "w", encoding="utf-8") as file:
        file.write(
            "PRIMARY DECISION FILE: screen_ofat_winners.csv\n\n"
            "A reference_is_winner=True entry means the reference coefficient "
            "had the lowest validation Macro-F1 error in that single-factor "
            "sweep, with the other coefficients held at their reference values.\n\n"
            "Do not claim a global optimum over the full Cartesian product. "
            "The full product contains 500 configurations, whereas this audit "
            "uses 16 reference-centered OFAT configurations.\n\n"
            "Do not rank configurations by val_total_loss because changing the "
            "loss coefficients changes the scale of that quantity. The script "
            "ranks by validation Macro-F1 error and uses motif classification "
            "cross-entropy only as a tie-breaker.\n\n"
            "The source test split is not evaluated or used for selection.\n"
        )

    print("\n[COMPLETE]")
    print("[RESULTS]", output_dir)
    print(
        "[READ FIRST]",
        os.path.join(output_dir, "screen_ofat_winners.csv"),
    )
    print("[INTERPRETATION]", final_note_path)


if __name__ == "__main__":
    run_sensitivity_audit()

## 14. Interpretation of the Retained Study Run

In the retained study run, `screen_ofat_winners.csv` reported
`reference_is_winner=True` for all four evaluated coefficients. The retained
configuration `(0.02, 0.20, 0.50, 0.70)` achieved a best validation Macro F1
of approximately `0.7160`, corresponding to a Macro-F1 error of approximately
`0.2840`.

Across the four single-factor sweeps, each retained coefficient yielded the
lowest validation Macro-F1 error when the remaining coefficients were held at
their reference values. The improvements over the nearest alternatives were
modest, ranging from approximately 0.15 to 0.39 percentage points. The results
therefore indicate a consistent, rather than large, validation advantage for
the retained configuration.

These results support the coefficients used in the final motif extractor. This
companion notebook documents the selection procedure referenced beside the
retained values in
`09_phase_initialized_sstkg_motif_extractor.ipynb`.

## 15. Generated Artifacts

```text
audit_manifest.json
all_trials.csv
screen_summary.csv
screen_ofat_winners.csv
screen_ofat_curves.csv
screen_<parameter>_sensitivity.png
HOW_TO_INTERPRET_RESULTS.txt
```

Optional confirmation outputs are added only when confirmation is enabled.